# Spark Session, schema configuration and csv data loading

## Verifying the CWD

In [0]:
import os
print(os.getcwd())

## Initiating the sparksession

In [0]:
from pyspark.sql import SparkSession

# Initiate spark session
spark = (
        SparkSession.
        builder.
        appName("SparkSession").
        getOrCreate()
)

## Defining an explicit schema for the data

I'm using [this dataset](https://www.kaggle.com/datasets/uradkr/alabama-sold-real-estate-intelligence-2026) for my analysis

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, BooleanType

alabama_real_estate_schema = StructType([
    StructField("type", StringType(), True),
    StructField("sub_type", StringType(), True),
    StructField("listPrice", FloatType(), True),
    StructField("lastSoldPrice", FloatType(), True),
    StructField("soldOn", StringType(), True),
    StructField("sqft", FloatType(), True),
    StructField("stories", FloatType(), True),
    StructField("beds", FloatType(), True),
    StructField("baths", FloatType(), True),
    StructField("baths_full", FloatType(), True),
    StructField("baths_full_calc", FloatType(), True),
    StructField("garage", FloatType(), True),
    StructField("year_built", FloatType(), True),
    StructField("postal_code", StringType(), True),
    StructField("is_valid_al_zip", BooleanType(), True),
    StructField("sold_year", FloatType(), True),
    StructField("sold_month", FloatType(), True),
    StructField("sold_quarter", FloatType(), True),
    StructField("property_age", FloatType(), True),
    StructField("sold_to_list_ratio", FloatType(), True),
    StructField("price_premium_pct", FloatType(), True),
    StructField("negotiation_outcome", StringType(), True),
    StructField("ratio_outlier_flag", BooleanType(), True),
    StructField("price_per_sqft", FloatType(), True),
    StructField("text_clean", StringType(), True),
    StructField("pii_redacted_flag", BooleanType(), True)
])

columns: list[str] = alabama_real_estate_schema.fieldNames() # Retrieve columns

In [0]:
print(columns)

## Reading the data

In [0]:
login:str = input("Please enter your login")

absolute_data_path = f"""/Workspace/Users/{login}@softserve.academy/lab1 - Databricks Fundamentals & DEV Setup/lab1/assignment/alabama_sold_real_estate_intelligence_2026.csv"""

## Making sure the provided absolute path does exist

In [0]:
# We're checking if the data path provided by the user exists or not - defensive programming
try:
    dbutils.fs.ls(absolute_data_path)
    print(f"Your path '{absolute_data_path}' exists. Reading the data from the csv...")
    
    alabama_df = spark.read.csv(absolute_data_path,
               schema=alabama_real_estate_schema, 
               header=True,
               ).select(columns)
                 
except Exception as e:
    raise FileNotFoundError(f"Your path {absolute_data_path} doesn't exist!")

# Loading the data

## Creating the schema

In [0]:
catalog = spark.catalog.currentCatalog()
schema = f"{catalog}.{login}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

## Specifying the path of the sourced table

In [0]:
table_name = "alabama_real_estate"
full_path = f"{schema}.{table_name}"

In [0]:
(alabama_df.write.
 format("delta").mode("overwrite").
 saveAsTable(full_path)
 )